# NB07: Internal training and evaluation (PCA -> Ridge -> Platt)

Trains the manuscript pipeline on TCGA OpenCLIP ViT-B/16 patient-mean
embeddings, evaluates on the held-out validation and test splits, and
produces Figure 3 panels A-D.

**Manuscript reference** (Methods, Results, Supp Tables S5/S8):
- Pipeline: StandardScaler -> PCA(k=384) -> Ridge(alpha=30.0) -> Platt scaling
- Trained on continuous scarHRD (regression target); Platt fitted to
  binary HRD_top20 (scarHRD >= 33) on the training split only
- Seed: 42 throughout
- Bootstrap reps (internal): 200
- TCGA test  AUROC = 0.766 (95% CI 0.727 - 0.803), AP = 0.423, Brier = 0.146
- TCGA val   AUROC = 0.775 (95% CI 0.739 - 0.808), AP = 0.432, Brier = 0.139
- HRD prevalence test = 21.2% (n_HRD+ = 177, n_HRD- = 656)

**Figure 3** panels:
- A: ROC test+val with 95% bootstrap CI shaded for test
- B: PR test+val with bootstrap CI shaded
- C: Calibration on test, 10 equal-width bins, bootstrap CI per bin
- D: KDE of predicted HRD probability split by HRD+ vs HRD- on test

**Required env**: `WORKSPACE`. **Inputs**: clean embeddings (NB06) + labels
(NB04+NB05). **Outputs**: `models/frozen_model.joblib`, `figures/fig3_*.{png,pdf}`,
`results/tcga_internal/{overall_metrics.csv,per_cancer_metrics.csv,bootstrap_curves.npz}`.

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    roc_curve, precision_recall_curve,
)
import matplotlib.pyplot as plt

# env
WORKSPACE = Path(os.environ.get("WORKSPACE", "./workspace")).resolve()
EMB_DIR = WORKSPACE / "embeddings"
LABELS_DIR = WORKSPACE / "labels"
MODELS_DIR = WORKSPACE / "models"
FIG_DIR = WORKSPACE / "figures"
RESULTS_DIR = WORKSPACE / "results" / "tcga_internal"
for d in [MODELS_DIR, FIG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# manuscript-locked params (Methods + Supp Table S5)
SEED = 42
PCA_N = 384
RIDGE_ALPHA = 30.0
HRD_THR = 33
TOP_FRAC = 0.20
BOOT_N = 200
ROC_GRID = 200       # number of FPR grid points for envelope
PR_GRID = 200        # recall grid points

print(f"SEED={SEED}, PCA_N={PCA_N}, RIDGE_ALPHA={RIDGE_ALPHA}, HRD_THR={HRD_THR}, BOOT_N={BOOT_N}")


In [ ]:
# load clean embeddings + labels, align on patient
emb_path = EMB_DIR / "patient_means_clean.parquet"
labels_path = LABELS_DIR / "labels.parquet"
assert emb_path.exists(), f"missing {emb_path}; run NB06 first"
assert labels_path.exists(), f"missing {labels_path}; run NB04+NB05 first"

X = pd.read_parquet(emb_path).set_index("patient")
labels = pd.read_parquet(labels_path)
labels["patient"] = labels["patient"].astype(str).str.upper().str.slice(0, 12)
labels = labels.drop_duplicates("patient").set_index("patient")

common = sorted(set(X.index) & set(labels.index))
X = X.loc[common]
labels = labels.loc[common].copy()

# require non-null HRD continuous score
mask_hrd = labels["HRD"].notna()
labels = labels.loc[mask_hrd].copy()
X = X.loc[labels.index]

print(f"aligned patients with HRD: {len(labels):,}")
print(f"feature dim              : {X.shape[1]}")
print(f"manuscript ref           : 8,109 patients, 512-d input")

# splits
idx_tr = labels.index[labels["split"] == "train"]
idx_va = labels.index[labels["split"] == "val"]
idx_te = labels.index[labels["split"] == "test"]
print(f"splits: train={len(idx_tr):,} val={len(idx_va):,} test={len(idx_te):,}")
print(f"manuscript ref          : 6,465 / 811 / 833")


In [ ]:
# verify HRD threshold reproduces manuscript value of 33 on training
y_tr_cont = labels.loc[idx_tr, "HRD"].astype(float).values
emp_thr = float(np.quantile(y_tr_cont, 1.0 - TOP_FRAC))
print(f"empirical top-{int(TOP_FRAC*100)}% threshold on training : {emp_thr:.2f}")
print(f"manuscript value                                          : {HRD_THR}")
print(f"using locked manuscript threshold                         : {HRD_THR}")

# binarize using the locked manuscript threshold
labels["HRD_top20"] = (labels["HRD"] >= HRD_THR).astype(int)

# train binary
y_tr_bin = labels.loc[idx_tr, "HRD_top20"].astype(int).values
n_pos_tr = int(y_tr_bin.sum())
print(f"training HRD+ : {n_pos_tr} / {len(y_tr_bin)} ({100*n_pos_tr/len(y_tr_bin):.1f}%)")


In [ ]:
# fit pipeline: StandardScaler -> PCA(384) -> Ridge(alpha=30.0)
X_tr = X.loc[idx_tr].values.astype(np.float32)
X_va = X.loc[idx_va].values.astype(np.float32)
X_te = X.loc[idx_te].values.astype(np.float32)

pipe = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("pca", PCA(n_components=PCA_N, random_state=SEED)),
    ("ridge", Ridge(alpha=RIDGE_ALPHA, random_state=SEED)),
])
pipe.fit(X_tr, y_tr_cont)

# Platt: LogReg on training ridge scores against binary HRD_top20
z_tr = pipe.predict(X_tr).reshape(-1, 1)
platt = LogisticRegression(max_iter=1000, solver="lbfgs", random_state=SEED)
platt.fit(z_tr, y_tr_bin)

# predict probs on val/test
def predict_prob(Xq):
    z = pipe.predict(Xq).reshape(-1, 1)
    return platt.predict_proba(z)[:, 1]

p_tr = predict_prob(X_tr)
p_va = predict_prob(X_va)
p_te = predict_prob(X_te)

y_va_bin = labels.loc[idx_va, "HRD_top20"].astype(int).values
y_te_bin = labels.loc[idx_te, "HRD_top20"].astype(int).values
print(f"predicted prob ranges: train [{p_tr.min():.3f},{p_tr.max():.3f}], "
      f"val [{p_va.min():.3f},{p_va.max():.3f}], test [{p_te.min():.3f},{p_te.max():.3f}]")


In [ ]:
# scalar bootstrap CIs for AUROC, AP, Brier
def bootstrap_ci(y, p, fn, n_boot=BOOT_N, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y)
    vals = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yb, pb = y[idx], p[idx]
        if yb.sum() == 0 or yb.sum() == n:
            vals[b] = np.nan
            continue
        try:
            vals[b] = fn(yb, pb)
        except Exception:
            vals[b] = np.nan
    return float(np.nanpercentile(vals, 2.5)), float(np.nanpercentile(vals, 97.5))

def metrics_with_ci(y, p, n_boot=BOOT_N):
    auc = float(roc_auc_score(y, p))
    ap = float(average_precision_score(y, p))
    brier = float(brier_score_loss(y, p))
    auc_lo, auc_hi = bootstrap_ci(y, p, roc_auc_score, n_boot)
    ap_lo, ap_hi = bootstrap_ci(y, p, average_precision_score, n_boot)
    return {
        "n": int(len(y)), "n_pos": int(y.sum()), "n_neg": int(len(y) - y.sum()),
        "AUROC": auc, "AUROC_lo": auc_lo, "AUROC_hi": auc_hi,
        "AP": ap, "AP_lo": ap_lo, "AP_hi": ap_hi,
        "Brier": brier,
    }

m_va = metrics_with_ci(y_va_bin, p_va)
m_te = metrics_with_ci(y_te_bin, p_te)

print("VALIDATION:")
print(f"  AUROC = {m_va['AUROC']:.3f} (95% CI {m_va['AUROC_lo']:.3f}-{m_va['AUROC_hi']:.3f})  ref 0.775 (0.739-0.808)")
print(f"  AP    = {m_va['AP']:.3f}  Brier = {m_va['Brier']:.3f}")
print()
print("TEST:")
print(f"  AUROC = {m_te['AUROC']:.3f} (95% CI {m_te['AUROC_lo']:.3f}-{m_te['AUROC_hi']:.3f})  ref 0.766 (0.727-0.803)")
print(f"  AP    = {m_te['AP']:.3f}  Brier = {m_te['Brier']:.3f}")

pd.DataFrame([{"split": "val", **m_va}, {"split": "test", **m_te}]).to_csv(
    RESULTS_DIR / "overall_metrics.csv", index=False
)


In [ ]:
# bootstrap envelope: pointwise 95% CI for ROC and PR curves on test
def roc_envelope(y, p, n_boot=BOOT_N, n_grid=ROC_GRID, seed=SEED):
    rng = np.random.default_rng(seed)
    fpr_grid = np.linspace(0.0, 1.0, n_grid)
    tprs = np.full((n_boot, n_grid), np.nan)
    n = len(y)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yb, pb = y[idx], p[idx]
        if yb.sum() == 0 or yb.sum() == n:
            continue
        fpr_b, tpr_b, _ = roc_curve(yb, pb)
        tprs[b] = np.interp(fpr_grid, fpr_b, tpr_b)
    return fpr_grid, np.nanpercentile(tprs, 2.5, axis=0), np.nanpercentile(tprs, 97.5, axis=0)

def pr_envelope(y, p, n_boot=BOOT_N, n_grid=PR_GRID, seed=SEED):
    rng = np.random.default_rng(seed)
    rec_grid = np.linspace(0.0, 1.0, n_grid)
    precs = np.full((n_boot, n_grid), np.nan)
    n = len(y)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yb, pb = y[idx], p[idx]
        if yb.sum() == 0 or yb.sum() == n:
            continue
        prec_b, rec_b, _ = precision_recall_curve(yb, pb)
        # precision_recall_curve returns rec descending; reverse for interp
        order = np.argsort(rec_b)
        precs[b] = np.interp(rec_grid, rec_b[order], prec_b[order])
    return rec_grid, np.nanpercentile(precs, 2.5, axis=0), np.nanpercentile(precs, 97.5, axis=0)

fpr_g, tpr_lo, tpr_hi = roc_envelope(y_te_bin, p_te)
rec_g, prec_lo, prec_hi = pr_envelope(y_te_bin, p_te)

np.savez(RESULTS_DIR / "bootstrap_curves.npz",
         fpr_grid=fpr_g, tpr_lo=tpr_lo, tpr_hi=tpr_hi,
         rec_grid=rec_g, prec_lo=prec_lo, prec_hi=prec_hi)
print("bootstrap envelopes saved")


In [ ]:
# per-cancer breakdown on test set
def per_cancer_table(idx, y, p, lbls):
    cancers = lbls.loc[idx, "cancer"].values
    rows = []
    for c in sorted(set(cancers)):
        m = (cancers == c)
        if m.sum() < 2 or y[m].sum() == 0 or y[m].sum() == m.sum():
            rows.append({"cancer": c, "n": int(m.sum()),
                         "n_pos": int(y[m].sum()),
                         "AUROC": np.nan, "AP": np.nan})
            continue
        rows.append({
            "cancer": c, "n": int(m.sum()),
            "n_pos": int(y[m].sum()),
            "AUROC": float(roc_auc_score(y[m], p[m])),
            "AP": float(average_precision_score(y[m], p[m])),
        })
    return pd.DataFrame(rows)

per_test = per_cancer_table(idx_te, y_te_bin, p_te, labels)
per_test = per_test.sort_values("n", ascending=False).reset_index(drop=True)
per_test.to_csv(RESULTS_DIR / "per_cancer_metrics.csv", index=False)
print(per_test.head(10).to_string(index=False))


In [ ]:
# save frozen model package for downstream notebooks (NB10, NB14, NB17, NB19)
frozen = {
    "pipe": pipe,
    "platt": platt,
    "feature_columns": list(X.columns),
    "feat_dim": int(X.shape[1]),
    "pca_n": PCA_N,
    "ridge_alpha": RIDGE_ALPHA,
    "threshold_HRD": HRD_THR,
    "seed": SEED,
    "manuscript_metrics": {
        "test_AUROC": 0.766, "test_AUROC_CI": (0.727, 0.803),
        "val_AUROC": 0.775, "val_AUROC_CI": (0.739, 0.808),
    },
    "this_run": {
        "test": m_te,
        "val": m_va,
    },
}
joblib.dump(frozen, MODELS_DIR / "frozen_model.joblib")
print(f"frozen model saved: {MODELS_DIR / 'frozen_model.joblib'}")


In [ ]:
# Fig 3A: ROC test + val with 95% bootstrap CI shaded for test
fpr_te, tpr_te, _ = roc_curve(y_te_bin, p_te)
fpr_va, tpr_va, _ = roc_curve(y_va_bin, p_va)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.fill_between(fpr_g, tpr_lo, tpr_hi, color="C0", alpha=0.20, label="test 95% CI")
ax.plot(fpr_te, tpr_te, color="C0", lw=2.0,
        label=f"test  AUROC = {m_te['AUROC']:.3f} ({m_te['AUROC_lo']:.3f}-{m_te['AUROC_hi']:.3f})")
ax.plot(fpr_va, tpr_va, color="C3", lw=1.5, ls="--",
        label=f"val   AUROC = {m_va['AUROC']:.3f} ({m_va['AUROC_lo']:.3f}-{m_va['AUROC_hi']:.3f})")
ax.plot([0, 1], [0, 1], color="0.6", lw=0.8, ls=":")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.005)
ax.set_title("Figure 3A: ROC, internal validation")
ax.legend(loc="lower right", fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3a_roc_test_val.png", dpi=300)
fig.savefig(FIG_DIR / "fig3a_roc_test_val.pdf")
plt.show()


In [ ]:
# Fig 3B: PR test + val with 95% bootstrap CI shaded for test
prec_te, rec_te, _ = precision_recall_curve(y_te_bin, p_te)
prec_va, rec_va, _ = precision_recall_curve(y_va_bin, p_va)
prev_te = float(y_te_bin.mean())

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.fill_between(rec_g, prec_lo, prec_hi, color="C2", alpha=0.20, label="test 95% CI")
ax.plot(rec_te, prec_te, color="C2", lw=2.0,
        label=f"test  AP = {m_te['AP']:.3f} ({m_te['AP_lo']:.3f}-{m_te['AP_hi']:.3f})")
ax.plot(rec_va, prec_va, color="C1", lw=1.5, ls="--",
        label=f"val   AP = {m_va['AP']:.3f} ({m_va['AP_lo']:.3f}-{m_va['AP_hi']:.3f})")
ax.axhline(prev_te, color="0.6", lw=0.8, ls=":", label=f"baseline = {prev_te:.3f}")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.005)
ax.set_title("Figure 3B: PR, internal validation")
ax.legend(loc="upper right", fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3b_pr_test_val.png", dpi=300)
fig.savefig(FIG_DIR / "fig3b_pr_test_val.pdf")
plt.show()


In [ ]:
# Fig 3C: calibration on test, 10 equal-width bins, bootstrap CI per bin
N_BINS = 10
edges = np.linspace(0.0, 1.0, N_BINS + 1)
centers = 0.5 * (edges[:-1] + edges[1:])

def bin_obs(y, p, edges):
    obs = np.full(len(edges) - 1, np.nan)
    for i in range(len(edges) - 1):
        m = (p >= edges[i]) & (p < edges[i + 1] if i < len(edges) - 2 else p <= edges[i + 1])
        if m.sum() > 0:
            obs[i] = float(y[m].mean())
    return obs

obs_te = bin_obs(y_te_bin, p_te, edges)

# bootstrap per-bin observed rate
rng = np.random.default_rng(SEED)
B = BOOT_N
boot_obs = np.full((B, N_BINS), np.nan)
n_te = len(y_te_bin)
for b in range(B):
    ix = rng.integers(0, n_te, size=n_te)
    boot_obs[b] = bin_obs(y_te_bin[ix], p_te[ix], edges)
lo = np.nanpercentile(boot_obs, 2.5, axis=0)
hi = np.nanpercentile(boot_obs, 97.5, axis=0)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
yerr = np.vstack([np.maximum(0, obs_te - lo), np.maximum(0, hi - obs_te)])
ax.errorbar(centers, obs_te, yerr=yerr, fmt="o-", color="C4", lw=1.5,
            markersize=5, capsize=2.5, label=f"test (n={n_te})")
ax.plot([0, 1], [0, 1], color="0.5", lw=0.8, ls=":", label="ideal")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed HRD frequency")
ax.set_title("Figure 3C: calibration, 10 equal-width bins")
ax.legend(loc="upper left", fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3c_calibration.png", dpi=300)
fig.savefig(FIG_DIR / "fig3c_calibration.pdf")
plt.show()


In [ ]:
# Fig 3D: KDE of predicted probability split by HRD+ vs HRD- on test
from scipy.stats import gaussian_kde
xs = np.linspace(0.0, 1.0, 400)
p_pos = p_te[y_te_bin == 1]
p_neg = p_te[y_te_bin == 0]

k_pos = gaussian_kde(p_pos, bw_method=0.20)
k_neg = gaussian_kde(p_neg, bw_method=0.20)

fig, ax = plt.subplots(figsize=(4.8, 4.5))
ax.fill_between(xs, k_neg(xs), color="C0", alpha=0.30,
                label=f"HRD- (n = {len(p_neg)})")
ax.fill_between(xs, k_pos(xs), color="C3", alpha=0.30,
                label=f"HRD+ (n = {len(p_pos)})")
ax.plot(xs, k_neg(xs), color="C0", lw=1.6)
ax.plot(xs, k_pos(xs), color="C3", lw=1.6)
ax.axvline(0.50, color="0.4", lw=0.8, ls="--", label="threshold = 0.50")
ax.set_xlim(0, 1)
ax.set_xlabel("Predicted HRD probability")
ax.set_ylabel("Density")
ax.set_title("Figure 3D: predicted probability distribution")
ax.legend(loc="upper right", fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3d_kde.png", dpi=300)
fig.savefig(FIG_DIR / "fig3d_kde.pdf")
plt.show()

print(f"manuscript ref: HRD+ n=177, HRD- n=656 (test set)")
print(f"this run     : HRD+ n={len(p_pos)}, HRD- n={len(p_neg)}")


In [ ]:
# final report
report = {
    "seed": SEED,
    "params": {
        "PCA_N": PCA_N, "RIDGE_ALPHA": RIDGE_ALPHA,
        "HRD_THR": HRD_THR, "BOOT_N": BOOT_N, "TOP_FRAC": TOP_FRAC,
    },
    "n_train": int(len(idx_tr)),
    "n_val": int(len(idx_va)),
    "n_test": int(len(idx_te)),
    "test_HRD_pos": int(y_te_bin.sum()),
    "test_HRD_neg": int(len(y_te_bin) - y_te_bin.sum()),
    "metrics": {"val": m_va, "test": m_te},
    "manuscript_refs": {
        "n_train": 6465, "n_val": 811, "n_test": 833,
        "test_HRD_pos": 177, "test_HRD_neg": 656,
        "test_AUROC": [0.766, [0.727, 0.803]],
        "val_AUROC":  [0.775, [0.739, 0.808]],
    },
    "frozen_model_path": str(MODELS_DIR / "frozen_model.joblib"),
}
(RESULTS_DIR / "report.json").write_text(json.dumps(report, indent=2, default=str))
print(json.dumps({k: report[k] for k in ["seed", "params", "n_test", "metrics", "manuscript_refs"]},
                 indent=2, default=str))
